## Initial Pre-processing

This notebook demonstrates my pre-processing steps for water main break data. I download and import the most recently available data from the [regional database](https://open-kitchenergis.opendata.arcgis.com/datasets/KitchenerGIS::water-main-breaks/about) and get rid of the unnecessary columns so we can focus on the features that are going to essential for our model later on.

Let's dive in!

In [ ]:
# Basic imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
%matplotlib inline

In [ ]:
# Import our most recent data and check out a sample
break_data = pd.read_csv("../data/raw/water_data.csv")
break_data.sample(10)

In [ ]:
break_data.shape

In [ ]:
break_data.info()

In [ ]:
break_data.columns

In [ ]:
def print_null_values(df, column=None):
    if column is None:
        # print null values for all columns
        for col in df.columns:
            null_values = df[col].isnull().sum()
            print(f"{col} - ", null_values)
    else:
        # print null values for a single column
        null_values = df[column].isnull().sum()
        print(f"{col} - ", null_values)

In [ ]:
# break_data.isna().sum()
print_null_values(break_data)

In [ ]:
break_data.columns

In [ ]:
break_data.drop(['WATBREAKINCIDENTID', 'ROAD_CLOSED', 'SIDEWALK_CLOSED', 'UNITS_IMPACTED', 'CW_SERVICE_REQUEST', 'STATUS_DATE',
                 'WORKORDER', 'RETURN_TO_NORMAL', 'REPAIR_TYPE', 'NEW_SECTION_LENGTH', 'MAINTENANCE_DESC', 'VALVES_CLOSED', 'VALVES_OPENED',
                 'HYDRANTS_CALLED_OUT', 'HYDRANTS_CALLED_BACK_IN', 'BACTERIA_TESTING_DATE', 'HEALTH_DEPT_NOTIFICATION', 'MOECC_SAC_NOTIFICATION',
                 'SAC_REFERENCE_NO', 'LOCAL_MOE_OFFICE', 'BWA_DWA', 'BWA_DWA_DECLARED', 'PROCEEDURES_FOLLOWED', 'RECORD_CHANGE_REQD', 'ASSET_DEPTH', 
                 'FROST_DEPTH', 'LINED_DATE', 'ACQUISITION', 'CONSULTANT', 'OWNERSHIP', 'BRIDGE_DETAILS', 'REL_CLEANING_AREA', 'REL_CLEANING_SUBAREA', 
                 'POSITIVE_PRESSURE_MAINTANED', 'AIR_GAP_MAINTANED', 'DISINFECTED', 'MECHANICAL_REMOVAL', 'FLUSHING_EXCAVATION', 'HIGHER_VELOCITY_FLUSHING',
                 'ANODE_INSTALLED', 'CIVIC_NUMBER', 'MAP_LABEL', 'LINED', 'LINED_MATERIAL', 'BRIDGE_MAIN', 'UNDERSIZED', 'SHALLOW_MAIN', 'OVERSIZED',
                 'CLEANED', 'Shape__Length', 'OBJECTID.1', 'ROADSEGMENTID.1', 'STATUS.1', 'ACQUISITION', 'CONSULTANT', 'OWNERSHIP', 'BRIDGE_DETAILS'], axis=1, inplace=True)

In [ ]:
break_data.shape

In [ ]:
break_data.head()

In [ ]:
print_null_values(break_data)

Let's investigate the types of values there are in the `BREAK_APPARENT_CAUSE` variable and `BREAK_NATURE` variable. There aren't a lot of missing values so I might be able to easily impute them.

In [ ]:
break_data.BREAK_APPARENT_CAUSE.unique()

In [ ]:
break_data.BREAK_APPARENT_CAUSE.value_counts()

I'll fill the nan values with 'UNKNOWN'

In [ ]:
break_data.BREAK_APPARENT_CAUSE.fillna('UNKNOWN', inplace=True)

In [ ]:
break_data.BREAK_APPARENT_CAUSE.value_counts()


In [ ]:
break_data.BREAK_NATURE.value_counts()

In [ ]:
break_data['BREAK_NATURE'] = break_data['BREAK_NATURE'].replace({'OTHER: WATER SERVICE': 'WATER SERVICE'})

In [ ]:
break_data.BREAK_NATURE.value_counts()

In [ ]:
break_data.BREAK_NATURE.unique()

In [ ]:
break_data.BREAK_NATURE.fillna('UNKNOWN', inplace=True)

In [ ]:
break_data.BREAK_NATURE.value_counts()

In [ ]:
break_data.BREAK_NATURE.unique()

In [ ]:
print_null_values(break_data)

In [ ]:
break_data.BREAK_CATEGORIZATION.value_counts()

In [ ]:
break_data.BREAK_CATEGORIZATION.unique()

In [ ]:
def fill_null_values(df, column, value):
    # fill null values in the specified column with the specified value
    df[column].fillna(value, inplace=True)

In [ ]:
# break_data.BREAK_CATEGORIZATION.fillna('UNKNOWN', inplace=True)
fill_null_values(break_data, 'BREAK_CATEGORIZATION', 'UNKNOWN')

In [ ]:
break_data.BREAK_CATEGORIZATION.value_counts()

In [ ]:
break_data['BREAK_NATURE'] = break_data['BREAK_NATURE'].replace({'OTHER': 'UNKNOWN'})

In [ ]:
break_data.BREAK_NATURE.unique()

In [ ]:
break_data.BREAK_NATURE.value_counts()

In [ ]:
print_null_values(break_data)

We can see there's some `STREET` names missing. It would be best to explore these missing attributes and see how we can possibly impute them. They still contain `LATITUDE` and `LONGITUDE` values so we can possibly identify them by those. Let's see what we can do...

In [ ]:
no_street = break_data.loc[break_data.STREET.isna()]
no_street

Looking at the latitude and longitude of the missing street addresses, there are actually only 5 distinct missing addresses. We can plot them on a map and see where they are located.

In [ ]:
import plotly
import plotly.express as px

In [ ]:
from config import token
api_token = token
px.set_mapbox_access_token(token)
fig = px.scatter_mapbox(data_frame=no_street, lat='latitude', lon='longitude')
fig.update_layout(mapbox_style="carto-positron", mapbox_accesstoken=token)
fig.show();

In [ ]:
# drop the rows with no street
break_data.dropna(subset=['STREET'], inplace=True)

In [ ]:
print_null_values(break_data)

In [ ]:
break_data.ASSET_SIZE.value_counts()

So there's 707 missing values for `ASSET_SIZE`. We could simply fill the null values with the mean value, the most frequent value, or just set them to 0. Another way that I feel might be the most appropriate approach is visualize on a map where these asset sizes occur, categorizing by street name, and then seeing if the null values occur on the same streets as some of those already known asset sizes, and then fill in the missing sizes with which neighbours their near.

In [ ]:
# pd.set_option("display.max_rows", None)
break_data.loc[break_data.ASSET_SIZE.isna()]

What I will try to do next is look at a sort of breakdown of the asset sizes, listed with their asset ID's and corresponding street names, and see if we can match up any non-null asset sizes from the same street as asset sizes with null values. There's a possibility of imputing the average asset size value from the same street for the null values. For now I am jusy hypothesizing this, I'm not entirely sure if it'll work but why not try right?

In [ ]:
asset_size = break_data[['STREET', 'ASSETID', 'ASSET_SIZE']]
# cherry picking null asset sizes to see which streets we could look at
asset_size.loc[asset_size.ASSET_SIZE.isna()].sample(10)

Next I'll see if there are any matching asset ID's for null and non-null asset sizes. If there are any matches, then it could be safe to say that I could impute the null values with the ID's matching asset size. If this doesn't prove to be true then I'll have to explore more options.

In [ ]:
asset_size[asset_size['STREET'] == 'FLORENCE AVE']

Unfortunately there are no matching ID's for this street. Let's check a few more streets to be sure that this trend might not hold...

In [ ]:
asset_size[asset_size['STREET'] == 'OTTAWA ST S']

In [ ]:
# display street names with no asset size
streets_with_na = asset_size.STREET[asset_size['ASSET_SIZE'].isna()].unique()
print(streets_with_na)

In [ ]:
asset_size[asset_size['STREET'] == 'WEBER ST E']

Well there we have it, each ID is unique to the asset size whether it's missing or not.

I will fill in the `NaN` values with the most frequent number (mode) for now.

In [ ]:
asset_size.ASSET_SIZE.value_counts()

In [ ]:
# fill asset size with the mode of the column
break_data['ASSET_SIZE'].fillna(break_data['ASSET_SIZE'].mode()[0], inplace=True)

In [ ]:
break_data.ASSET_SIZE.value_counts()

In [ ]:
# drop the rows of asset sizes that are 0.0
break_data.drop(break_data[break_data['ASSET_SIZE'] == 0.0].index, inplace=True)

In [ ]:
break_data.ASSET_SIZE.value_counts()

In [ ]:
print_null_values(break_data)

In [ ]:
break_data.ASSET_YEAR_INSTALLED.nunique()

In [ ]:
# compare the asset year installed with the installation date
break_data[['ASSET_YEAR_INSTALLED', 'INSTALLATION_DATE', 'INCIDENT_DATE']].sample(10)

In [ ]:
# convert asset year installed to datetime
break_data['ASSET_YEAR_INSTALLED'] = pd.to_datetime(break_data['ASSET_YEAR_INSTALLED'], format='%Y')
break_data[['ASSET_YEAR_INSTALLED', 'INSTALLATION_DATE', 'INCIDENT_DATE']].sample(10)

In [ ]:
# how many incident dates are less than asset year installed
break_data.loc[break_data['INCIDENT_DATE'] < break_data['ASSET_YEAR_INSTALLED']]

In [ ]:
# drop the rows where the incident date is less than the asset year installed
break_data.drop(break_data[break_data['INCIDENT_DATE'] < break_data['ASSET_YEAR_INSTALLED']].index, inplace=True)

In [ ]:
break_data.shape

In [ ]:
# show the missing asset year installed rows with their installation date
break_data[['ASSET_YEAR_INSTALLED', 'INSTALLATION_DATE']][break_data['ASSET_YEAR_INSTALLED'].isna()].sample(30)

In [ ]:
print_null_values(break_data)

From reading different studies, it seems as though the year the pipe was installed or rather the age of the pipe is a critical factor in predicting breaks/time of failure. In this case, I believe I may have to drop the observations where no year is indicated. 

In [ ]:
break_data.dropna(axis=0, subset=['ASSET_YEAR_INSTALLED'], inplace=True)

In [ ]:
print(break_data.shape)
print_null_values(break_data)

In [ ]:
for col in break_data.columns:
    if break_data[col].isnull().sum() > 0:
        print(col, break_data[col].unique())

Pretty sure if we drop the installation dates that are missing instead of imputing them, that will get rid of all of the rest of the missing information...

In [ ]:
break_data.dropna(axis=0, subset=['INSTALLATION_DATE'], inplace=True)

In [ ]:
print_null_values(break_data)

In [ ]:
print(break_data.shape)
break_data.sample(15)

Lucky for us that took care of our missing values in the `ASSET_MATERIAL` column. Now we have a clean dataset finally to do some EDA and feature engineering.

In [ ]:
# show the rows where the incident date is less than the asset year installed
break_data.loc[break_data['INCIDENT_DATE'] < break_data['ASSET_YEAR_INSTALLED']]

In [ ]:
break_data.columns

In [ ]:
break_data.to_csv('../data/interim/cleaned_break_data.csv', index=False)